# Training — DPO / Cal-DPO / SimPO

Run this notebook **separately for each method**:
1. Set `METHOD` (and optionally `RUN_SWEEP`) in the config cell
2. Run all cells
3. Sweep results, best config, and the trained model are all saved to Drive and downloaded

Requires `01_setup.ipynb` to have been run first.

In [ ]:
# ── Edit these before running ────────────────────────────────────────────────
METHOD       = 'dpo'   # 'dpo' | 'caldpo' | 'simpo'
RUN_SWEEP    = True    # False → skip sweep and use default hyperparameters
SWEEP_EPOCHS = 2       # epochs per sweep candidate (keep low for speed)
FULL_EPOCHS  = 5       # epochs for the final best-config training run

# Hyperparameter grids (from instruction §3 / §4)
SWEEPS = {
    'dpo':    [{'beta': b} for b in [1e-3, 2e-3, 3e-3, 1e-2, 1e-1]],
    'caldpo': [{'beta': b} for b in [1e-3, 2e-3, 3e-3, 1e-2, 1e-1]],
    'simpo':  [{'beta': b, 'gamma': g}
               for b in [2.0, 2.5] for g in [0.5, 1.0, 1.5]],
}
DEFAULTS = {
    'dpo':    {'beta': 1e-3},
    'caldpo': {'beta': 1e-3},
    'simpo':  {'beta': 2.0, 'gamma': 1.0},
}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q 'transformers>=4.40' datasets 'trl>=0.8'

In [ ]:
import os, json, time
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import GPT2LMHeadModel, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from trl.experimental.cpo import CPOTrainer, CPOConfig  # confirmed path per trl docs

ROOT     = '/content/drive/MyDrive/cal-simpo-outputs'
SFT_DIR  = f'{ROOT}/sft_model'
PREF_DIR = f'{ROOT}/pref_data'
CKPT_DIR = f'{ROOT}/checkpoints'
RES_DIR  = f'{ROOT}/results'
for d in [f'{CKPT_DIR}/{METHOD}', RES_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
PROMPT_LEN = 32
MAX_SEQ    = 192
torch.manual_seed(42)

tok = AutoTokenizer.from_pretrained(SFT_DIR)
tok.pad_token    = tok.eos_token
tok.padding_side = 'right'
print(f'Method: {METHOD}  |  Device: {DEVICE}  |  Sweep: {RUN_SWEEP}')

In [ ]:
train_p = json.load(open(f'{PREF_DIR}/train.json'))
val_p   = json.load(open(f'{PREF_DIR}/val.json'))
keys    = ('prompt', 'chosen', 'rejected')
train_ds = Dataset.from_list([{k: p[k] for k in keys} for p in train_p])
val_ds   = Dataset.from_list([{k: p[k] for k in keys} for p in val_p])
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# Cal-DPO loss override — defined here so it's available to make_trainer()
class CalDPOTrainer(DPOTrainer):
    """DPO + calibration loss (Xiao et al. 2024, Algorithm 1)."""
    def dpo_loss(self, policy_chosen_logps, policy_rejected_logps,
                 reference_chosen_logps, reference_rejected_logps):
        r_w = policy_chosen_logps   - reference_chosen_logps
        r_l = policy_rejected_logps - reference_rejected_logps
        bt  = -F.logsigmoid(r_w - r_l)   # no β inside sigmoid
        cal = (F.mse_loss(r_w, torch.full_like(r_w,  0.5 / self.beta)) +
               F.mse_loss(r_l, torch.full_like(r_l, -0.5 / self.beta)))
        return bt + cal, r_w.detach(), r_l.detach()


def best_margin(log_history):
    """Highest eval reward margin seen during a training run."""
    for key in ('eval_rewards/margins', 'rewards/margins'):
        vals = [e[key] for e in log_history if key in e]
        if vals:
            return max(vals)
    return None


def make_trainer(config, output_dir, epochs, save=False):
    """
    Build a trainer for METHOD with the given hyperparameter config.
    save=True  → keep checkpoints and load best at end (for final run).
    save=False → no checkpoints saved (for sweep candidates).
    Returns (trainer, models_to_delete).
    """
    # trl only exposes eval_loss to Trainer's checkpoint selection —
    # rewards/margins is logged but not in the eval metrics dict.
    save_args = dict(
        save_strategy='steps', save_steps=50, save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
    ) if save else dict(
        save_strategy='no',
        load_best_model_at_end=False,
    )

    common = dict(
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,   # effective batch 32
        learning_rate=5e-6, optim='rmsprop', warmup_steps=30,
        fp16=(DEVICE == 'cuda'), seed=42,
        logging_steps=10,
        eval_strategy='steps', eval_steps=50,
        report_to='none',
        **save_args,
    )

    policy = GPT2LMHeadModel.from_pretrained(SFT_DIR)

    if METHOD == 'dpo':
        ref = GPT2LMHeadModel.from_pretrained(SFT_DIR)
        trainer = DPOTrainer(
            model=policy, ref_model=ref, processing_class=tok,
            args=DPOConfig(
                output_dir=output_dir,
                beta=config['beta'],
                loss_type=['sigmoid'],   # list form required in trl >= 0.8
                # max_prompt_length removed in trl >= 0.8; data is pre-truncated in step 3
                max_length=MAX_SEQ,
                **common,
            ),
            train_dataset=train_ds, eval_dataset=val_ds,
        )
        return trainer, [policy, ref]

    elif METHOD == 'caldpo':
        ref = GPT2LMHeadModel.from_pretrained(SFT_DIR)
        # loss_type not set — CalDPOTrainer overrides dpo_loss() directly
        trainer = CalDPOTrainer(
            model=policy, ref_model=ref, processing_class=tok,
            args=DPOConfig(
                output_dir=output_dir,
                beta=config['beta'],
                max_length=MAX_SEQ,
                **common,
            ),
            train_dataset=train_ds, eval_dataset=val_ds,
        )
        return trainer, [policy, ref]

    else:  # simpo
        trainer = CPOTrainer(
            model=policy, processing_class=tok,
            args=CPOConfig(
                output_dir=output_dir,
                loss_type='simpo', cpo_alpha=0.0,
                beta=config['beta'], simpo_gamma=config['gamma'],
                max_length=MAX_SEQ,
                **common,
            ),
            train_dataset=train_ds, eval_dataset=val_ds,
        )
        return trainer, [policy]

In [ ]:
# ── Hyperparameter sweep ──────────────────────────────────────────────────────
# Each candidate config is trained for SWEEP_EPOCHS; best margin wins.
# If RUN_SWEEP=False, this cell is skipped and the default config is used.

sweep_log = []   # one entry per candidate: {config, best_margin}

if RUN_SWEEP:
    configs = SWEEPS[METHOD]
    print(f'Sweeping {len(configs)} configs for {METHOD} ({SWEEP_EPOCHS} epochs each)...')

    for i, cfg in enumerate(configs):
        print(f'\n[{i+1}/{len(configs)}] config={cfg}')
        trainer, models = make_trainer(
            cfg,
            output_dir=f'{CKPT_DIR}/{METHOD}/sweep_{i}',
            epochs=SWEEP_EPOCHS,
            save=False,
        )
        trainer.train()
        margin = best_margin(trainer.state.log_history)
        print(f'  best margin = {margin}')
        sweep_log.append({'config': cfg, 'best_margin': margin,
                          'log_history': trainer.state.log_history})
        # Clean up before next candidate
        for m in models: del m
        del trainer;  torch.cuda.empty_cache()

    # Pick config with highest margin
    best_cfg = max(sweep_log, key=lambda x: x['best_margin'] or -1e9)['config']
    print(f'\nBest config: {best_cfg}')
else:
    best_cfg = DEFAULTS[METHOD]
    print(f'Skipping sweep. Using default config: {best_cfg}')

In [ ]:
# ── Final training run with best config ──────────────────────────────────────
# Trains for FULL_EPOCHS; saves best checkpoint (by reward margin) to Drive.

print(f'Final run: {METHOD}  config={best_cfg}  epochs={FULL_EPOCHS}')
final_dir = f'{CKPT_DIR}/{METHOD}/final'

trainer, models = make_trainer(
    best_cfg,
    output_dir=f'{CKPT_DIR}/{METHOD}/run_final',
    epochs=FULL_EPOCHS,
    save=True,
)

t0 = time.time()
trainer.train()
training_time_s = time.time() - t0
print(f'Training time: {training_time_s / 60:.1f} min')

# save_model() saves the best checkpoint (load_best_model_at_end=True)
trainer.save_model(final_dir)
tok.save_pretrained(final_dir)

# Save log_history for 03_eval_plots.ipynb (figures use this for training dynamics)
json.dump(trainer.state.log_history,
          open(f'{RES_DIR}/{METHOD}_log.json', 'w'), indent=2)

# Capture which step was best and what the metric value was
best_step   = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric

for m in models: del m
del trainer;  torch.cuda.empty_cache()
print(f'Model saved to {final_dir}')
print(f'Best step: {best_step}  |  Best metric: {best_metric}')

In [ ]:
# ── Save sweep results and best config to JSON ───────────────────────────────

# Strip log_history from sweep_log for a compact summary file
sweep_summary = [
    {'config': e['config'], 'best_margin': e['best_margin']}
    for e in sweep_log
]

best_config_record = {
    'method':           METHOD,
    'best_config':      best_cfg,
    'best_margin':      best_metric,
    'best_checkpoint':  best_step,
    'training_time_s':  training_time_s,   # wall-clock seconds for final run
    'sweep_ran':        RUN_SWEEP,
    'sweep_epochs':     SWEEP_EPOCHS if RUN_SWEEP else None,
    'full_epochs':      FULL_EPOCHS,
    'fixed_args': {
        'learning_rate': 5e-6,
        'optim':         'rmsprop',
        'warmup_steps':  30,
        'effective_batch': 32,
        'max_seq_len':   MAX_SEQ,
        'prompt_len':    PROMPT_LEN,
    },
}

json.dump(sweep_log,          open(f'{RES_DIR}/{METHOD}_sweep_full.json',    'w'), indent=2)
json.dump(sweep_summary,      open(f'{RES_DIR}/{METHOD}_sweep_summary.json', 'w'), indent=2)
json.dump(best_config_record, open(f'{RES_DIR}/{METHOD}_best_config.json',   'w'), indent=2)

print(f'Saved:')
print(f'  {METHOD}_sweep_summary.json  — best margin per candidate')
print(f'  {METHOD}_sweep_full.json     — full log_history per candidate')
print(f'  {METHOD}_best_config.json    — winning config + training args + time')

In [ ]:
# ── Download ──────────────────────────────────────────────────────────────────
# Model is already on Drive. This gives you a local copy too.
import shutil
from google.colab import files

# Zip and download the trained model
model_zip = f'/content/{METHOD}_model'
shutil.make_archive(model_zip, 'zip', final_dir)
files.download(f'{model_zip}.zip')

# Zip and download the JSON result files for this method
results_zip = f'/content/{METHOD}_results'
shutil.make_archive(results_zip, 'zip', RES_DIR,
                    base_dir='.')   # includes all *_{method}_*.json files
files.download(f'{results_zip}.zip')

print(f'Downloads started: {METHOD}_model.zip and {METHOD}_results.zip')